# SQL DataBase

In [8]:
import duckdb
import pandas as pd

# Loading CVS into DuckDB
See associate sql database file (database.sql) to get a better understanding of the tables

In [4]:
connection = duckdb.connect(database=':memory:')

connection.execute("""
CREATE TABLE housing_data AS
SELECT * FROM read_csv_auto('../data/MedianPricesofExistingDetachedHomesHistoricalData - Median Price.csv', header=True, skip=1)

""")

connection.execute("""
CREATE TABLE population AS
SELECT * FROM read_csv_auto('../data/population_clean.csv', header=True)
""")

connection.execute("""
CREATE TABLE debt AS
SELECT * FROM read_csv_auto('../data/debt_2003_2025 _clean.csv', header=True)
""")



# Query debt, housing, and population tables 

In [5]:
# Querying debt data from 2003 to 2025

connection.execute("""
SELECT
    CAST('20' || SUBSTR(quarter, 1, 2) AS INT) AS year,
    quarter,
    Mortgage,
    "Credit Card"   AS credit_card,
    "Student Loan"  AS student_loan,
    "Auto Loan"     AS auto_loan,
    "HE Revolving"  AS he_revolving
FROM debt
WHERE CAST('20' || SUBSTR(quarter, 1, 2) AS INT) BETWEEN 2003 AND 2025;
""").fetchdf()


# avg debt by year

connection.execute("""
SELECT
    CAST('20' || SUBSTR(quarter, 1, 2) AS INT) AS year,
    AVG(Mortgage)       AS avg_mortgage,
    AVG("Credit Card")  AS avg_credit_card,
    AVG("Student Loan") AS avg_student_loan,
    AVG("Auto Loan")    AS avg_auto_loan,
    AVG("HE Revolving") AS avg_he_revolving
FROM debt
GROUP BY year
ORDER BY year;
""").fetchdf()




,year,avg_mortgage,avg_credit_card,avg_student_loan,avg_auto_loan,avg_he_revolving
0,2003,5.215000,0.69250,0.245000,0.660000,0.267500
1,2004,6.095000,0.70750,0.300000,0.735000,0.400000
2,2005,6.805000,0.72500,0.375000,0.780000,0.535000
3,2006,7.870000,0.74500,0.450000,0.807500,0.592500
4,2007,8.790000,0.80500,0.525000,0.810000,0.627500
5,2008,9.262500,0.85500,0.605000,0.805000,0.685000
6,2009,8.995000,0.81750,0.687500,0.742500,0.710000
7,2010,8.647500,0.74000,0.777500,0.705000,0.680000
8,2011,8.432500,0.69500,0.857500,0.720000,0.632500
9,2012,8.100000,0.67500,0.935000,0.760000,0.582500


In [ ]:
# querying housing data --> TODO Need to fix and work on further
# df = pd.read_csv('../data/MedianPricesofExistingDetachedHomesHistoricalData - Median Price.csv', header=1)
# df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# connection.register('housing_data_df', df)
# connection.execute("CREATE TABLE housing_clean AS SELECT * FROM housing_data_df")
# connection.execute("SELECT * FROM housing_clean").fetchdf()

# connection.execute("SELECT * FROM housing_data").fetchdf()

list(connection.execute("SELECT * FROM housing_data LIMIT 1").fetchdf().columns)


['column0']

In [ ]:
# Querying population data
connection.execute("""
SELECT * FROM population
WHERE year BETWEEN 2000 AND 2020
ORDER BY year;
""").fetchdf()

# population growth rate comparing each year to previous year
connection.execute("""
SELECT
                   year,
                   population,
                   LAG(population) OVER (ORDER BY year) AS previous_population,
                   ROUND(((population - LAG(population) OVER (ORDER BY year)) * 100.0) / LAG(population) OVER (ORDER BY year), 2) AS growth_rate_percentage
FROM population 
WHERE year BETWEEN 2000 AND 2020
ORDER BY year;
""").fetchdf()

,year,population,previous_population,growth_rate_percentage
0,2000,33987.977,NaN,NaN
1,2001,34479.458,33987.977,1.45
2,2002,34871.843,34479.458,1.14
3,2003,35253.159,34871.843,1.09
4,2004,35574.576,35253.159,0.91
5,2005,35827.943,35574.576,0.71
6,2006,36021.202,35827.943,0.54
7,2007,36250.311,36021.202,0.64
8,2008,36604.337,36250.311,0.98
9,2009,36961.229,36604.337,0.97
